# Train YOLOv8 (feces + pig) on Colab GPU

Trains the 2-class **feces** / **pig** YOLOv8 detector on a free Google
Colab/T4 GPU, evaluates it against the **mAP50 >= 0.80** target, and copies a
downloadable `training-outputs.zip` back to your mounted Google Drive.

The detector is 2-class on purpose. The source Roboflow export annotates both
`feces` and `pig`; keeping the pig boxes means the model no longer has to
treat pig bodies as ambiguous background, and the verify server can report a
pig-only frame as a genuine "no feces" instead of a false positive.

### Run order
1. **Runtime -> Change runtime type -> GPU (T4)**.
2. Run all cells in order.
3. When finished, download `training-outputs.zip` from Drive (path printed at the end)
   and install it with `python training/install_colab_weights.py <path>`.


In [ ]:
# 1) Mount Google Drive (open the link, approve, paste the code)
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted.")


In [ ]:
# 2) Configuration
import os

# Dataset zip you uploaded to Drive (edit if in a subfolder).
# Use the CONVERTED 2-class layout (dataset/{images,labels}/{train,val}).
# Regenerate it locally with:  python training/pack_colab_zip.py
# Do NOT use the raw Roboflow export, and do NOT use a single-class zip:
# this notebook trains against training/dataset.yaml which declares BOTH
# classes (feces = 0, pig = 1). A single-class zip trains without any error
# and leaves the pig class unsupervised, which is what caused false-positive
# sprays. Cell 4 asserts both class ids are present and fails fast if not.
DRIVE_ZIP = "/content/drive/MyDrive/pigfeces-yolo.zip"
assert os.path.exists(DRIVE_ZIP), "Upload pigfeces-yolo.zip to Drive first."

# Where the final outputs zip is written back to Drive.
DRIVE_OUT = "/content/drive/MyDrive/training-outputs.zip"

# Repo URL to clone the training scripts (yields train.py/evaluate.py/dataset.yaml).
# The dataset itself is git-ignored, so it is NOT in git - the zip above provides it.
REPO_URL = "https://github.com/renumbracode/feces-detector-ai"
assert REPO_URL, "Set REPO_URL to your repo (any public/fork)."

print("OK")

In [ ]:
# 3) Clone the repo to fetch training/ scripts + dataset.yaml
import os
repo = "/content/feces-detector-ai"
if os.path.exists(repo) and os.listdir(repo):
    print("repo already present:", repo)
else:
    !git clone {REPO_URL} {repo}
assert os.path.exists(f"{repo}/training/train.py"), "train.py not found after clone"
print("Scripts ready.")


In [ ]:
# 4) Unzip the dataset into $REPO/dataset
import os, zipfile, glob
repo = "/content/feces-detector-ai"
os.makedirs(f"{repo}/dataset", exist_ok=True)
with zipfile.ZipFile(DRIVE_ZIP) as z:
    z.extractall(f"{repo}/dataset")
base = f"{repo}/dataset"
for d in ["images/train","images/val","labels/train","labels/val"]:
    p = f"{base}/{d}"
    print(f"{d}:", len(os.listdir(p)) if os.path.isdir(p) else "MISSING")

# --- Fail fast if this is not the 2-class dataset -------------------------
# The model is trained against training/dataset.yaml (nc: 2, feces + pig).
# A single-class zip trains happily against that yaml and produces a model
# that has never seen a pig, so it cannot learn to reject one. Verify both
# class ids are actually present before spending 40-90 GPU-minutes.
import yaml
cfg = yaml.safe_load(open(f"{repo}/training/dataset.yaml"))
names = cfg["names"]
if isinstance(names, dict):
    expected = {int(k) for k in names}
else:
    expected = set(range(len(names)))
print("dataset.yaml classes:", names)

per_class = {i: 0 for i in range(max(expected) + 1)}
n_boxes = 0
for split in ("train", "val"):
    for lbl in glob.glob(f"{base}/labels/{split}/*.txt"):
        for line in open(lbl):
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) < 5:
                continue
            cls = int(float(parts[0]))
            if cls not in expected:
                raise SystemExit(
                    f"ABORT: {lbl} has class id {cls}, not in dataset.yaml {sorted(expected)}. "
                    "This zip does not match the training config."
                )
            per_class[cls] += 1
            n_boxes += 1

for i in sorted(per_class):
    print(f"  class {i} ({names[i] if not isinstance(names, dict) else names[str(i)]}): {per_class[i]} boxes")
print("total boxes:", n_boxes)

missing = [i for i in sorted(expected) if per_class[i] == 0]
if missing:
    raise SystemExit(
        f"ABORT: class id(s) {missing} have ZERO boxes. dataset.yaml declares "
        f"{names}, so this zip is single-class or incomplete. Regenerate it with "
        "`python training/pack_colab_zip.py` and re-upload. Continuing here would "
        "train a model that never learns to reject a pig."
    )
print("OK: both classes present. Proceeding.")

In [ ]:
# 5) Point dataset.yaml path: at the Colab dataset location
import os, re, pathlib
repo = "/content/feces-detector-ai"
DATA_YAML = f"{repo}/training/dataset.yaml"
text = pathlib.Path(DATA_YAML).read_text()
abs_path = f"{repo}/dataset".replace("\\", "/")
text = re.sub(r"path:\s*\S+", f"path: {abs_path}", text)
pathlib.Path(DATA_YAML).write_text(text)
print(text)


In [ ]:
# 6) Install dependencies (ultralytics pulls torch + CUDA)
!pip -q install ultralytics
import torch, ultralytics
print("ultralytics", ultralytics.__version__)
print("torch", torch.__version__)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## Train

100 epochs @ 640, batch 16. On a free Colab T4 expect **~40-90 minutes**
(early-stopping patience 30 may finish sooner). Leave this cell running until it completes.


In [ ]:
# 7) TRAIN (long step)
repo = "/content/feces-detector-ai"
!python {repo}/training/train.py \
      --model yolov8n.pt \
      --epochs 100 \
      --imgsz 640 \
      --batch 16 \
      --patience 30 \
      --project {repo}/training/runs \
      --name detect \
      --device 0 \
      --exist-ok

# Resolve where the weights actually landed. With --exist-ok ultralytics
# writes straight to runs/detect/, but an incremented run name nests under
# runs/detect/train/. Prefer the first, fall back to the second, so every
# cell below works either way.
import os
RUN_DIR = f"{repo}/training/runs/detect"
if not os.path.isdir(f"{RUN_DIR}/weights"):
    if os.path.isdir(f"{RUN_DIR}/train/weights"):
        RUN_DIR = f"{RUN_DIR}/train"
    else:
        raise FileNotFoundError(f"no weights dir under {RUN_DIR}")
BEST = f"{RUN_DIR}/weights/best.pt"
print("run dir :", RUN_DIR)
print("best.pt :", BEST, os.path.exists(BEST))

In [ ]:
# 8) EVALUATE against the 0.80 target
repo = "/content/feces-detector-ai"
!python {repo}/training/evaluate.py \
      --model {BEST} \
      --data {repo}/training/dataset.yaml \
      --target 0.80 \
      --device 0
print("Evaluate done. See evaluation_summary.txt above.")

In [ ]:
# 9) Package outputs and copy back to Drive
import os, zipfile, shutil
repo = "/content/feces-detector-ai"
print("run contents:", os.listdir(RUN_DIR))

zpath = "/content/training-outputs.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    zw = f"{RUN_DIR}/weights"
    for name in ("best.pt", "last.pt"):
        p = f"{zw}/{name}"
        if os.path.exists(p):
            z.write(p, f"weights/{name}")
    # metrics
    for name in ("results.csv", "evaluation_summary.txt"):
        p = f"{RUN_DIR}/{name}"
        if os.path.exists(p):
            z.write(p, name)
    # a couple diagnostic plots if present
    for name in os.listdir(RUN_DIR):
        if name in ("labels.jpg", "results.png", "confusion_matrix.png", "PR_curve.png"):
            z.write(f"{RUN_DIR}/{name}", name)

shutil.copy(zpath, DRIVE_OUT)
print("DOWNLOAD READY ->", DRIVE_OUT)
print("Install locally with: python training/install_colab_weights.py", DRIVE_OUT)

## After training

Install the outputs so the weights land in the layout the verify server expects:

```bash
python training/install_colab_weights.py /path/to/training-outputs.zip
```

That writes `training/runs/detect/weights/best.pt`, refuses to clobber an
existing model without `--force`, and prints the `YOLO_MODEL` line to use.
Then start the verify server and the dashboard.

### If the score lands below 0.80

The target is measured on Roboflow frames. The device serves frames from your
own ESP32-S3 pig-pen camera, so a high Roboflow mAP and a good field result are
different claims — see `training/notes/` for the honest val set.
